In [ ]:
import pandas as pd
df = pd.read_csv("network_events.csv")



In [ ]:
port_counts = df.groupby("src_ip")["port"].nunique()
threshold = 2
suspicious_src = port_counts[port_counts > threshold].index
df["suspicious_try"] = df["src_ip"].isin(suspicious_src)


srcip = df.groupby("src_ip")["dst_ip"].nunique()
too_much = 2
suspicious_ip = srcip[srcip >= too_much].index
df["suspicious_ip"] = df["src_ip"].isin(suspicious_ip)


bytes_num = df.groupby("src_ip")["bytes"].sum()
too_muchh = 9000
suspicious_bytes = bytes_num[bytes_num > too_muchh].index
df["suspicious_bytes"] = df["src_ip"].isin(suspicious_bytes)


spread = df.groupby(["src_ip", "port"])["dst_ip"].nunique()
too_muchhh = 3
suspicious_pair = spread[spread >= too_muchhh].index
df["suspicious_spread"] = pd.Series(list(zip(df["src_ip"], df["port"]))).isin(suspicious_pair)


many_src = df.groupby("dst_ip")["src_ip"].nunique()
too_muchhhh = 3
suspicious_target = many_src[many_src >= too_muchhhh].index
df["suspicious_target"] = df["dst_ip"].isin(suspicious_target)


df["suspicious_score"] = (df["suspicious_try"].astype(int) + df["suspicious_ip"].astype(int) + df["suspicious_bytes"].astype(int) + df["suspicious_spread"].astype(int) + df["suspicious_target"].astype(int))
df[["src_ip", "dst_ip", "bytes", "port", "suspicious_try", "suspicious_ip", "suspicious_bytes", "suspicious_spread", "suspicious_target", "suspicious_score"]]


df["risk_level"] = "low"
df.loc[df["suspicious_score"] == 0, "risk_level"] = "normal"
df.loc[df["suspicious_score"] == 1, "risk_level"] = "low"
df.loc[df["suspicious_score"] == 2, "risk_level"] = "medium"
df.loc[df["suspicious_score"] == 3, "risk_level"] = "high"
df.loc[df["suspicious_score"] == 4, "risk_level"] = "critical"


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

df[["src_ip", "dst_ip", "bytes", "port", "suspicious_try", "suspicious_ip", "suspicious_bytes", "suspicious_spread", "suspicious_target", "suspicious_score", "risk_level"]]




In [ ]:
df["src_event_count"] = df.groupby("src_ip")["src_ip"].transform("count")
df["bytes_sum"] = df.groupby("src_ip")["bytes"].transform("sum")
df["avg_bytes_per_event"] = (df["bytes_sum"] // df["src_event_count"])
df["unique_ports_per_src_ip"] = df.groupby("src_ip")["port"].transform("nunique")
df["unique_dst_count"] = df.groupby("src_ip")["dst_ip"].transform("nunique")

df[["src_ip", "bytes", "src_event_count", "bytes_sum", "avg_bytes_per_event", "max_bytes_per_event", "unique_ports_per_src_ip", "unique_dst_count"]].drop_duplicates(subset="src_ip")


In [ ]:
df["attack_type"] = "normal"


df.loc[
    (df["attack_type"] == "normal") &
    (df["max_bytes_per_event"] >= 8000) &
    (df["bytes_sum"] >= 15000) &
    (df["unique_dst_count"] <= 2),
    "attack_type"
] = "data_exfiltration"


df.loc[
    (df["attack_type"] == "normal") &
    (df["unique_dst_count"] >= 4) &
    (df["unique_ports_per_src_ip"] <= 2) &
    (df["avg_bytes_per_event"] <= 500) &
    (df["src_event_count"] >= 4),
    "attack_type"
] = "network_scan"


df.loc[
    (df["attack_type"] == "normal") &
    (df["unique_ports_per_src_ip"] >= 4),
    "attack_type"
] = "port_scan"


df.loc[
    (df["attack_type"] == "normal") &
    (df["src_event_count"] >= 6) &
    (df["unique_dst_count"] == 1) &
    (df["avg_bytes_per_event"] >= 500),
    "attack_type"
] = "brute_force"


df.loc[
    (df["attack_type"] == "normal") &
    (df["src_event_count"] >= 6) &
    (df["unique_dst_count"] == 1) &
    (df["unique_ports_per_src_ip"] == 1) &
    (df["avg_bytes_per_event"] <= 300),
    "attack_type"
] = "beaconing"



df[["src_ip","dst_ip","port","bytes","attack_type"]]

In [ ]:
reason_map = {
    "port_scan": [
        "Many unique ports contacted by the same source IP",
        "Low average bytes per connection"
    ],
    "network_scan": [
        "Many unique destination IPs contacted",
        "Low bytes per event",
        "Multiple events from the same source"
    ],
    "brute_force": [
        "Repeated attempts to a single destination",
        "Moderate traffic size per event"
    ],
    "beaconing": [
        "Frequent small periodic connections",
        "Same destination and port"
    ],
    "data_exfiltration": [
        "Extremely large data transfer",
        "High total bytes sent",
        "Limited destination spread"
    ]
}

df["reasons"] = ""
df["reasons"] = df["attack_type"].map(reason_map)
pd.set_option("display.max_colwidth", None)
df[["src_ip","dst_ip","port","bytes","attack_type", "reasons"]]

In [ ]:
def build_alert(row):
    # skip non-attacks
    if not isinstance(row["reasons"], list):
        return ""

    # format attack name (network_scan → Network Scan)
    attack = row["attack_type"].replace("_", " ").title()

    # convert reasons list → bullet points
    reasons_text = "\n".join(f"- {r}" for r in row["reasons"])

    # build final message
    return f"""ALERT: {attack} detected from {row['src_ip']}

Reasons:
{reasons_text}"""

# apply row-wise
df["alert_message"] = df.apply(build_alert, axis=1)






for i, msg in enumerate(df["alert_message"]):
    if msg:
        print(msg)
        print("------")
    if i > 10:
        break